In [0]:
-- ============================================================================
-- MegaMart Databricks Capstone
-- Stage: 05 - Inventory Analysis
--
-- Purpose:
--   Analyze inventory position, sales velocity, and supplier delivery
--   performance using the cleaned Silver layer.
--
-- Source:
--   megamart_dev.silver.*
--
-- Target:
--   megamart_dev.gold.*
--
-- Generator-specific decisions:
--   1. Inventory contains one snapshot per store/product combination.
--   2. Products contain supplier_id, so product -> supplier is available.
--   3. Purchase orders do not contain product_id or quantity_ordered.
--   4. Purchase orders do not contain a promised delivery date/SLA.
--
-- Therefore:
--   - stockout/low-stock analysis is based on current quantity_on_hand.
--   - sales velocity is based on historical sales quantity.
--   - supplier lead time is based on delivered purchase orders.
--   - delivery completion rate is used instead of inventing an on-time SLA.
--   - reorder-point calculation is handled by the following Python stage.
-- ============================================================================


-- ============================================================================
-- 1. Select Project Catalog
-- ============================================================================

USE CATALOG megamart_dev;


-- ============================================================================
-- 2. Source Data Verification
-- ============================================================================

SELECT
    'inventory' AS dataset,
    COUNT(*) AS record_count
FROM silver.inventory

UNION ALL

SELECT
    'sales_transactions' AS dataset,
    COUNT(*) AS record_count
FROM silver.sales_transactions

UNION ALL

SELECT
    'products' AS dataset,
    COUNT(*) AS record_count
FROM silver.products

UNION ALL

SELECT
    'suppliers' AS dataset,
    COUNT(*) AS record_count
FROM silver.suppliers

UNION ALL

SELECT
    'purchase_orders' AS dataset,
    COUNT(*) AS record_count
FROM silver.purchase_orders;


-- ============================================================================
-- 7. Inventory Analysis Verification
-- ============================================================================

SELECT
    'inventory_status' AS view_name,
    COUNT(*) AS record_count
FROM gold.inventory_status

UNION ALL

SELECT
    'product_performance' AS view_name,
    COUNT(*) AS record_count
FROM gold.product_performance

UNION ALL

SELECT
    'supplier_scorecard' AS view_name,
    COUNT(*) AS record_count
FROM gold.supplier_scorecard;


-- ============================================================================
-- 8. Stock Status Summary
-- ============================================================================

SELECT
    stock_status,
    COUNT(*) AS store_product_count

FROM gold.inventory_status

GROUP BY
    stock_status

ORDER BY
    stock_status;


-- ============================================================================
-- 9. Velocity Summary
-- ============================================================================

SELECT
    velocity_class,
    COUNT(*) AS store_product_count

FROM gold.product_performance

GROUP BY
    velocity_class

ORDER BY
    velocity_class;


-- ============================================================================
-- 10. Out-of-Stock and Low-Stock Products
-- ============================================================================

SELECT
    store_id,
    store_name,
    product_id,
    product_name,
    category,
    quantity_on_hand,
    stock_status,
    supplier_id,
    supplier_name

FROM gold.inventory_status

WHERE stock_status IN (
    'OUT_OF_STOCK',
    'LOW_STOCK'
)

ORDER BY
    stock_priority DESC,
    quantity_on_hand ASC,
    store_id,
    product_id;


-- ============================================================================
-- 11. Fast-Moving Products
-- ============================================================================

SELECT
    overall_velocity_rank,
    store_id,
    store_name,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    average_daily_sales,
    quantity_on_hand,
    estimated_days_of_inventory,
    velocity_class

FROM gold.product_performance

WHERE velocity_class = 'FAST_MOVING'

ORDER BY
    overall_velocity_rank;


-- ============================================================================
-- 12. Slow-Moving / Potential Overstock Candidates
-- ============================================================================

SELECT
    store_id,
    store_name,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    average_daily_sales,
    quantity_on_hand,
    estimated_days_of_inventory,
    velocity_class

FROM gold.product_performance

WHERE velocity_class = 'SLOW_MOVING'
  AND quantity_on_hand >= 50

ORDER BY
    average_daily_sales ASC,
    quantity_on_hand DESC;


-- ============================================================================
-- 13. Supplier Performance
-- ============================================================================

SELECT
    supplier_id,
    supplier_name,
    total_purchase_orders,
    delivered_orders,
    failed_or_incomplete_orders,
    delivery_reliability_pct,
    average_lead_time_days,
    minimum_lead_time_days,
    maximum_lead_time_days,
    reliability_definition

FROM gold.supplier_scorecard

ORDER BY
    delivery_reliability_pct DESC,
    average_lead_time_days ASC,
    supplier_id;
    

-- ============================================================================
-- 14. Store-Level Inventory Summary
-- ============================================================================

SELECT
    store_id,
    store_name,

    COUNT(*) AS total_products,

    SUM(
        CASE
            WHEN stock_status = 'OUT_OF_STOCK'
            THEN 1
            ELSE 0
        END
    ) AS out_of_stock_products,

    SUM(
        CASE
            WHEN stock_status = 'LOW_STOCK'
            THEN 1
            ELSE 0
        END
    ) AS low_stock_products,

    SUM(
        CASE
            WHEN stock_status = 'HEALTHY'
            THEN 1
            ELSE 0
        END
    ) AS healthy_stock_products,

    SUM(quantity_on_hand)
        AS total_quantity_on_hand

FROM gold.inventory_status

GROUP BY
    store_id,
    store_name

ORDER BY
    store_id;


-- ============================================================================
-- 15. Audit Pipeline Run
-- ============================================================================

INSERT INTO audit.pipeline_runs
SELECT
    CONCAT(
        '05_inventory_',
        DATE_FORMAT(
            CURRENT_TIMESTAMP(),
            'yyyyMMddHHmmss'
        )
    ) AS run_id,

    'megamart_pipeline' AS pipeline_name,

    '05_inventory_analysis' AS stage_name,

    'SUCCESS' AS status,

    CURRENT_TIMESTAMP() AS start_time,

    CURRENT_TIMESTAMP() AS end_time,

    (
        SELECT COUNT(*)
        FROM silver.inventory
    ) AS records_processed,

    0 AS records_failed,

    NULL AS error_message;


-- ============================================================================
-- End of Stage 05 Inventory Analysis
-- ============================================================================